In [5]:
import sqlite3

# Crear o conectar a la base de datos
conn = sqlite3.connect("alumnos.db")
cursor = conn.cursor()

# Crear tabla si no existe
cursor.execute("""
CREATE TABLE IF NOT EXISTS persona (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    nombre TEXT NOT NULL,
    imagen TEXT NOT NULL,
    embedding TEXT NOT NULL
)
""")

conn.commit()
conn.close()

print("✅ Base de datos y tabla creadas con éxito.")

✅ Base de datos y tabla creadas con éxito.


In [6]:
import os
import face_recognition
import json

ruta_rostros = "rostros"
datos_personas = []

# Leer todos los archivos de la carpeta
for archivo in os.listdir(ruta_rostros):
    if archivo.lower().endswith((".jpg", ".jpeg", ".png")):
        ruta_imagen = os.path.join(ruta_rostros, archivo)
        imagen = face_recognition.load_image_file(ruta_imagen)
        ubicaciones = face_recognition.face_locations(imagen)
        
        if len(ubicaciones) != 1:
            print(f"⚠️  {archivo}: se encontraron {len(ubicaciones)} rostros. Se omite.")
            continue
        
        # Extraer embeddings
        embedding = face_recognition.face_encodings(imagen, known_face_locations=ubicaciones)[0]
        
        # Extraer nombre desde el nombre del archivo
        nombre_base = os.path.splitext(archivo)[0]
        nombre = ''.join([c if c.isalpha() else ' ' for c in nombre_base])
        nombre = ' '.join(nombre.split())  # Limpiar espacios dobles
        
        # Convertir el embedding a lista
        embedding_lista = embedding.tolist()
        
        datos_personas.append({
            "nombre": nombre,
            "imagen": archivo,
            "embedding": json.dumps(embedding_lista)
        })

print(f"✅ Procesamiento completo: {len(datos_personas)} personas detectadas.")

⚠️  AlejandroOliden.jpg: se encontraron 2 rostros. Se omite.
✅ Procesamiento completo: 33 personas detectadas.


In [7]:
import sqlite3

conn = sqlite3.connect("alumnos.db")
cursor = conn.cursor()

insertados = 0
omitidos = 0

for persona in datos_personas:
    nombre = persona["nombre"]
    imagen = persona["imagen"]
    embedding = persona["embedding"]
    
    # Verificar si ya existe por nombre o imagen
    cursor.execute("SELECT COUNT(*) FROM persona WHERE nombre = ? OR imagen = ?", (nombre, imagen))
    existe = cursor.fetchone()[0]
    
    if existe == 0:
        cursor.execute("INSERT INTO persona (nombre, imagen, embedding) VALUES (?, ?, ?)", (nombre, imagen, embedding))
        insertados += 1
    else:
        print(f"🟡 Persona omitida (duplicado): {nombre} / {imagen}")
        omitidos += 1

conn.commit()
conn.close()

print(f"✅ Inserción completada. Insertados: {insertados}, Omitidos: {omitidos}")

✅ Inserción completada. Insertados: 33, Omitidos: 0


In [8]:
import sqlite3
import json

conn = sqlite3.connect("alumnos.db")
cursor = conn.cursor()

cursor.execute("SELECT id, nombre, imagen, embedding FROM persona LIMIT 5")
registros = cursor.fetchall()

for id_, nombre, imagen, embedding in registros:
    embedding_vector = json.loads(embedding)
    print(f"🧑 ID: {id_} | Nombre: {nombre} | Imagen: {imagen}")
    print(f"🔢 Embedding (primeros 5 valores): {embedding_vector[:5]}\n")

conn.close()

🧑 ID: 1 | Nombre: AdrianCisneros | Imagen: AdrianCisneros.jpg
🔢 Embedding (primeros 5 valores): [-0.18842539191246033, 0.06261848658323288, 0.010994374752044678, -0.040993139147758484, -0.08247467130422592]

🧑 ID: 2 | Nombre: AngelOncoy | Imagen: AngelOncoy.jpg
🔢 Embedding (primeros 5 valores): [-0.09755001217126846, 0.15744857490062714, 0.028419021517038345, -0.06143449619412422, -0.14083674550056458]

🧑 ID: 3 | Nombre: AnthonyLezcano | Imagen: AnthonyLezcano.jpg
🔢 Embedding (primeros 5 valores): [-0.18411098420619965, 0.13990691304206848, 0.043012771755456924, -0.02195819467306137, -0.09124263375997543]

🧑 ID: 4 | Nombre: BensonHilario | Imagen: BensonHilario.jpg
🔢 Embedding (primeros 5 valores): [-0.06999474763870239, 0.11319024860858917, -0.014425810426473618, -0.04836982116103172, -0.10704396665096283]

🧑 ID: 5 | Nombre: BryanMarin | Imagen: BryanMarin.jpg
🔢 Embedding (primeros 5 valores): [-0.11136632412672043, 0.09577270597219467, 0.012086605653166771, 0.032366685569286346, -0.0

In [16]:
import face_recognition
import sqlite3
import os
import json
import numpy as np
from numpy.linalg import norm

# 🔧 Umbral de similitud (menor = más estricto)
umbral_similitud = 0.45

# Función para calcular distancia euclidiana
def calcular_distancia(v1, v2):
    return norm(np.array(v1) - np.array(v2))

# Función principal para insertar si no hay duplicado
def insertar_persona_si_nueva(ruta_imagen):
    if not os.path.exists(ruta_imagen):
        print("❌ La imagen no existe.")
        return

    # Cargar imagen y detectar rostro
    imagen = face_recognition.load_image_file(ruta_imagen)
    ubicaciones = face_recognition.face_locations(imagen)
    
    if len(ubicaciones) != 1:
        print(f"⚠️ La imagen debe contener exactamente un rostro. Se encontraron: {len(ubicaciones)}")
        return

    # Obtener embedding
    embedding_nuevo = face_recognition.face_encodings(imagen, known_face_locations=ubicaciones)[0]
    embedding_lista = embedding_nuevo.tolist()

    # Extraer nombre desde el nombre del archivo
    archivo = os.path.basename(ruta_imagen)
    nombre_base = os.path.splitext(archivo)[0]
    nombre = ''.join([c if c.isalpha() else ' ' for c in nombre_base])
    nombre = ' '.join(nombre.split())

    # Conectar a la base de datos
    conn = sqlite3.connect("alumnos.db")
    cursor = conn.cursor()
    cursor.execute("SELECT id, nombre, imagen, embedding FROM persona")
    registros = cursor.fetchall()

    # Comparar con embeddings existentes
    for id_, nombre_bd, imagen_bd, emb_texto in registros:
        emb_bd = json.loads(emb_texto)
        distancia = calcular_distancia(embedding_lista, emb_bd)
        if distancia < umbral_similitud:
            print(f"🟡 Duplicado detectado: '{nombre}' se parece a '{nombre_bd}' (distancia = {distancia:.4f})")
            conn.close()
            return

    # Insertar si no es duplicado
    cursor.execute("INSERT INTO persona (nombre, imagen, embedding) VALUES (?, ?, ?)",
                   (nombre, archivo, json.dumps(embedding_lista)))
    conn.commit()
    conn.close()
    print(f"✅ {nombre} insertado correctamente en la base de datos.")


In [17]:
insertar_persona_si_nueva("rostros/PedroPerez.jpg")

🟡 Duplicado detectado: 'PedroPerez' se parece a 'JaimeLescano' (distancia = 0.2899)


In [19]:
insertar_persona_si_nueva("rostros/CristianCueva.jpg")

🟡 Duplicado detectado: 'CristianCueva' se parece a 'JuanCarranza' (distancia = 0.4045)
